In [1]:
# Cell 1: Install dependencies (uncomment if you haven't installed geemap yet)
!pip install earthengine-api geemap geedim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 116.4 MB/s eta 0:00:0000:01


In [2]:
import ee
import geemap

# -------------------------------------------------------------------------
# 1. Initialize Google Earth Engine
# -------------------------------------------------------------------------
try:
    ee.Initialize(project='constant-carver-465702-b1')  # Replace 'your-project-id' with your actual GCP project ID
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='constant-carver-465702-b1')  # Replace 'your-project-id' with your actual GCP project ID

print("Earth Engine successfully initialized.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine successfully initialized.


In [3]:
# -------------------------------------------------------------------------
# 2. Define AOI & Filter Sentinel-1 Collection
# -------------------------------------------------------------------------
# Hawke's Bay Coastal & River Region (Esk Valley / Napier)
aoi = ee.Geometry.BBox(176.60, -39.60, 177.10, -39.00)

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(aoi)
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.eq('relativeOrbitNumber_start', 81)))  # Ascending; orbit 73 only covered ~10% of the AOI

# -------------------------------------------------------------------------
# 3. Select Pre- and Post-Event Acquisitions
# -------------------------------------------------------------------------
# One pass is delivered as several GRD slices. .first() kept only one slice,
# which left most of the AOI (including Napier / Esk Valley) without data,
# so mosaic every slice from the pass instead.
def acquisition(start, end):
    images = s1.filterDate(start, end)
    print(f"{start} to {end}: {images.size().getInfo()} slice(s)")
    for scene_id in images.aggregate_array('system:index').getInfo():
        print("   ", scene_id)
    print("    Relative orbit(s):", images.aggregate_array('relativeOrbitNumber_start').distinct().getInfo())
    return images.mosaic().clip(aoi)

pre_image = acquisition('2023-01-21', '2023-01-22')
post_image = acquisition('2023-02-14', '2023-02-15')  # 14 Feb ~8pm NZDT, during peak flooding

# Verify each acquisition covers the AOI before exporting
def aoi_coverage(image):
    has_data = image.select('VV').mask().gt(0).unmask(0, False).rename('has_data')  # False: count pixels outside the swath as 0
    return has_data.reduceRegion(ee.Reducer.mean(), aoi, scale=100, maxPixels=1e9).get('has_data').getInfo()

for name, image in [('Pre-event', pre_image), ('Post-event', post_image)]:
    pct = 100 * aoi_coverage(image)
    print(f"{name} AOI coverage: {pct:.1f}%")
    if pct < 95:
        print("    WARNING: this pass does not cover the whole AOI. Shrink the AOI or use a "
              "different relative orbit / pass direction before exporting.")


2023-01-21 to 2023-01-22: 2 slice(s)
    S1A_IW_GRDH_1SDV_20230121T070708_20230121T070737_046878_059F11_F7A2
    S1A_IW_GRDH_1SDV_20230121T070737_20230121T070802_046878_059F11_60E1
    Relative orbit(s): [81]
2023-02-14 to 2023-02-15: 2 slice(s)
    S1A_IW_GRDH_1SDV_20230214T070707_20230214T070736_047228_05AAC4_8E2A
    S1A_IW_GRDH_1SDV_20230214T070736_20230214T070801_047228_05AAC4_9F82
    Relative orbit(s): [81]
Pre-event AOI coverage: 100.0%
Post-event AOI coverage: 100.0%


In [4]:
# Which Sentinel-1 passes cover the AOI around the event?
candidates = (ee.ImageCollection('COPERNICUS/S1_GRD')
              .filterBounds(aoi)
              .filterDate('2023-01-20', '2023-03-01')
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')))

def add_coverage(image):
    has_data = image.select('VV').mask().gt(0).unmask(0).rename('has_data')
    pct = has_data.reduceRegion(ee.Reducer.mean(), aoi, scale=200, maxPixels=1e9).get('has_data')
    return image.set('aoi_coverage', pct)

rows = (candidates.map(add_coverage)
        .reduceColumns(ee.Reducer.toList(4),
                       ['system:index', 'orbitProperties_pass', 'relativeOrbitNumber_start', 'aoi_coverage'])
        .get('list').getInfo())

print(f"{'Date (UTC)':<17} {'Pass':<11} {'Orbit':>5} {'AOI coverage':>13}")
for scene_id, pass_dir, orbit, cov in sorted(rows, key=lambda r: r[0][17:32]):
    print(f"{scene_id[17:32]:<17} {pass_dir:<11} {orbit:>5} {100 * cov:12.1f}%")


Date (UTC)        Pass        Orbit  AOI coverage
20230120T173020   DESCENDING     73         91.4%
20230121T070708   ASCENDING      81         99.0%
20230121T070737   ASCENDING      81         99.9%
20230127T172224   DESCENDING    175        100.0%
20230128T065921   ASCENDING       8        100.0%
20230201T173020   DESCENDING     73         91.2%
20230208T172223   DESCENDING    175        100.0%
20230209T065920   ASCENDING       8        100.0%
20230213T173031   DESCENDING     73         90.5%
20230214T070707   ASCENDING      81         98.9%
20230214T070736   ASCENDING      81        100.0%
20230220T172223   DESCENDING    175        100.0%
20230221T065920   ASCENDING       8        100.0%
20230225T173019   DESCENDING     73         91.2%
20230226T070707   ASCENDING      81         98.9%
20230226T070736   ASCENDING      81         99.9%


In [5]:
# -------------------------------------------------------------------------
# 4. Speckle Reduction & Difference Surface
# -------------------------------------------------------------------------
smoothing_radius = 30  # meters
pre_vv = pre_image.select('VV').focalMean(smoothing_radius, 'circle', 'meters')
post_vv = post_image.select('VV').focalMean(smoothing_radius, 'circle', 'meters')
pre_vh = pre_image.select('VH').focalMean(smoothing_radius, 'circle', 'meters')
post_vh = post_image.select('VH').focalMean(smoothing_radius, 'circle', 'meters')

# Calculate log-ratio difference (dB)
vv_diff = post_vv.subtract(pre_vv).rename('vv_diff')
vh_diff = post_vh.subtract(pre_vh).rename('vh_diff')

# -------------------------------------------------------------------------
# 5. Permanent Water Masking & Flood Baseline Thresholding
# -------------------------------------------------------------------------
jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
# occurrence is masked wherever water was never observed, so unmask to 0 first;
# otherwise baseline_mask is nodata over most land
permanent_water = jrc_water.select('occurrence').unmask(0).gt(80)
land_area_mask = permanent_water.Not()

flood_threshold = -3.5  # dB drop threshold
flood_mask = vv_diff.lt(flood_threshold).And(land_area_mask).rename('baseline_mask')

# -------------------------------------------------------------------------
# 6. Combine Stack & Cast Uniformly to Float32
# -------------------------------------------------------------------------
export_stack = ee.Image.cat([
    pre_vv.rename('pre_vv'),
    pre_vh.rename('pre_vh'),
    post_vv.rename('post_vv'),
    post_vh.rename('post_vh'),
    vv_diff.rename('vv_diff'),
    flood_mask.rename('baseline_mask').float()  # Convert byte mask to float32
]).toFloat()

# -------------------------------------------------------------------------
# 7. Download & Save to Google Drive
# -------------------------------------------------------------------------
import os
import shutil
from google.colab import drive

# The stack is kept on Google Drive so it survives a Colab runtime reset
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/Coastal-Change-Detection'
os.makedirs(DRIVE_DIR, exist_ok=True)

tif_name = 'CycloneGabrielle_HawkesBay_S1_Stack_v2.tif'
local_filename = os.path.join('/content', tif_name)  # Download target on the Colab disk
output_filename = os.path.join(DRIVE_DIR, tif_name)  # Final location on Drive

print("\nStarting download...")
geemap.download_ee_image(
    export_stack,
    filename=local_filename,
    region=aoi,
    scale=10,
    crs='EPSG:32760',  # UTM zone 60S, so 10 m pixels are true metres
)

# Move the finished file off the Colab disk and onto Drive
if os.path.exists(local_filename):
    shutil.move(local_filename, output_filename)

import numpy as np
import rasterio

# Nodata is written as -inf; check every band actually has data before preprocessing
if os.path.exists(output_filename):
    print(f"Saved '{output_filename}'. Data per band:")
    with rasterio.open(output_filename) as src:
        for i, name in enumerate(src.descriptions, start=1):
            band = src.read(i)
            print(f"    {name:<14} valid {100 * np.isfinite(band).mean():5.1f}%, exact zeros {100 * (band == 0).mean():5.1f}%")
else:
    print("Download failed — no file written.")


Starting direct local download...


/usr/local/lib/python3.13/dist-packages/geemap/common.py:12333: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/252 tiles [00:00<?]

/usr/local/lib/python3.13/dist-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


Saved 'CycloneGabrielle_HawkesBay_S1_Stack_v2.tif'. Data per band:
    pre_vv         valid  99.7%, exact zeros   0.0%
    pre_vh         valid  99.7%, exact zeros   0.0%
    post_vv        valid  99.7%, exact zeros   0.0%
    post_vh        valid  99.7%, exact zeros   0.0%
    vv_diff        valid  99.7%, exact zeros   0.0%
    baseline_mask  valid  99.7%, exact zeros  98.3%
